# Mutual Information Scaling in Image Datasets

This notebook provides an overview of the experiments and analysis in [Convy et al. (2022)](https://iopscience.iop.org/article/10.1088/2632-2153/ac44a9) regarding mutual information scaling in image datasets. The code used throughout this notebook is adapted from the `image.py` and `mine.py` modules. 

## A brief introduction to mutual information estimation

This work focuses on a quantity called _mutual information_ (MI), which describes the dependence between two variables $a$ and $b$ obeying the joint probability distribution $p(a, b)$. It is defined as the difference between the entropy of $p(a)p(b)$, which is the product of the marginal distributions, and the entropy of $p(a, b)$:

\begin{equation}
\text{MI}(a,b) = \text{S}[p(a)p(b)] - \text{S}[p(a, b)] = \text{S}[p(a)] + \text{S}[p(b)] - \text{S}[p(a,b)],
\end{equation}

where $\text{S}$ is the usual Shannon entropy. The MI can be qualitatively understood as a generalization of the Pearson correlation coefficient to non-linear relationships between variables, and it serves as the most general measure of dependence. This means that a non-zero correlation coefficient always implies a non-zero MI value, but the converse is not necessarily true.

Given access to the underlying probability distributions $p(a, b)$, $p(a)$, and $p(b)$, it is usually straightforward to compute the individual entropies and thus the MI. However, what if we only had access to $N$ samples from the joint distribution $p(a, b)$? If the variables are discrete and span a relatively small number of values, then the entropy could be estimated from $\text{S}[p(x)] = \sum_x p(x)\log p(x)$ using the observed frequencies in place of the probabilities. However, in cases where the variable $x$ is continuous or effectively continuous, the observed frequencies of each value in the domain will not carry sufficient information for the entropies and thus the MI to be estimated. 

To get around this issue, we need to impose some functional form on $p(a, b)$ and $p(a)p(b)$. One powerful method is to represent the probability distributions using a neural network trained on the $N$ samples from $p(a, b)$. This can be done in a straightforward manner by constructing artificial samples from the product-of-marginal distribution $p(a)p(b)$, and then optimizing the neural network model using a cross-entropy loss function to distinguish between samples from $p(a)p(b)$ and samples from $p(a, b)$. The general workflow is:

1. From samples $\{(a_i, b_i)\}^N_{i=1}$, generate a new set $\{(a_i, b_j)\}_{i \neq j}$ by randomly shuffling values for the variable $b$ between different samples. This simulates sampling from $p(a)p(b)$.
2. Train a logistic regression neural network algorithm to distinguish between samples taken from the two datasets in the previous step.
3. Use the output of the neural network to estimate the MI of $\{(a_i, b_i)\}^N_{i=1}$.

In our case, we will be interested in the MI between different sets of pixels in an image dataset, so our $a$ and $b$ variables will correspond to different groups of pixels in the same image. The shuffling in step 1 will consists of stitching together new images using pixel patches taken from the original samples.

## Testing on Gaussian Markov random fields

To ensure that our algorithm works properly, we can test it on data samples from distributions with known MI values. The most convenient class of distribution for this purpose is the Gaussian Markov random field, which is a multivariate Gaussian distribution parameterized by its _precision matrix_. The precision matrix is simply the inverse of the covariance matrix, and its off-diagonal elements determine the conditional correlation between the corresponding Gaussian variables. The following code shows functions which generate the precision matrix for three different MI patterns:

In [ ]:
import numpy as np

def get_area_law_prec(length, rho):
    q = np.eye(length**2)
    for i in range(length):
        for j in range(length):
            q_row = i * length + j
            for (m, l) in [(i + 1, j), (i - 1, j), (i, j - 1), (i, j + 1)]:
                if (m < length and m >= 0) and (l < length and l >= 0):
                    q_col = m * length + l
                    q[q_row, q_col] = rho
    return q

def get_diffuse_volume_prec(length, rho):
    q = np.full([length**2, length**2], rho)
    for i in range(length**2):
        q[i, i] = 1
    return q

def get_sparse_volume_prec(length, rho):
    gen = np.random.RandomState(123456789)
    q = np.eye(length**2)
    for i in range(length):
        for j in range(length):
            q_row = i * length + j
            for (m, l) in [(i + 1, j), (i - 1, j), (i, j - 1), (i, j + 1)]:
                if (m < length and m >= 0) and (l < length and l >= 0):
                    q_col = m * length + l
                    q[q_row, q_col] = rho
    shuffle = gen.permutation(length**2)
    q = q[shuffle, :]
    q = q[:, shuffle]
    return q

The `get_area_law_prec` function creates a precision matrix in which variables are conditionally correlated with only their four nearest neighbors when arranged on a 2D grid. The `get_diffuse_volume_prec` function, by contrast, constructs a precision matrix in which every variable is equally correlated with every other variable, regardless of their positions on the grid. Finally, the `get_sparse_volume_prec` creates a precision matrix that is a mix of the other two, in that it starts out by generating nearest-neighbor correlations but then randomizes the position of the variables on the grid. This results in each variable being correlated with four other variables at random locations on the grid.

Using samples from these Gaussian distributions (we can set the means arbitrarily to zero, since it does not affect the MI value), we can now train our neural network classifier. Because we are interested in capturing the MI structures of the different distributions, we will use a fully-connected neural network rather than, say, a convolutional network to avoid introducing spatial biases into the model. The code used to build the model is given below: 

In [ ]:
import ast

from tensorflow import keras as ks

class Model():
    def __init__(self, image_shape, settings):
        self.drop = float(settings['drop'])
        self.learn_rate = float(settings['learn'])
        self.layers = ast.literal_eval(settings['layers'])
        self.build_model(image_shape)

    def build_model(self, image_shape):
        joint_input = ks.Input(shape = image_shape)
        marginal_input = ks.Input(shape = image_shape)
        model_core = ks.models.Sequential()
        model_core.add(ks.Input(shape = image_shape))
        model_core.add(ks.layers.Flatten())
        for layer_size in self.layers:
            model_core.add(ks.layers.Dense(layer_size, activation = 'relu'))
            model_core.add(ks.layers.Dropout(self.drop))
        model_core.add(ks.layers.Dense(1, activation = None))
        joint_output = model_core(joint_input)
        marginal_output = model_core(marginal_input)
        self.model = ks.Model(inputs = [joint_input, marginal_input], outputs = [joint_output, marginal_output])

The model structure is fairly standard, with the exception of the inputs and outputs being split based on the kind of image being fed in. This has been done simply to make the model easier to work with when doing MI estimation, and is not necessary.

The following code trains the model on samples from the specified Gaussian Markov random field. The size of the "image" (i.e. the grid of variables) is set to 28 x 28, which matches the size of the real image datasets that we will be working with later on. The samples are separated into two regions, a square set of variables in the middle of the grid and the remaining set of pixels surrounding it. The collective states of these two pixel patches represent the variables $a$ and $b$ whose correlation we will be aiming to estimate. 

In [ ]:
import numpy as np

from src import mine, image as img

image_type = "diffuse" # Set to "area". "diffuse", or "sparse".
strength = "large" # Set to "small" or "large"
num_images = 700000
inner_length = 10
batch_size = 64
max_epochs = 100
model_settings = dict(
    drop = 0, 
    learn = 1e-4, 
    layers = "[256, 256]", 
    patience = 20, 
    optm = "rms")

rho = img.rho_values[image_type][strength]
print(f"Sampling {num_images} images...")
(images, _, _) = img.get_images(image_type, num_images, strength)
(_, height, width) = images.shape
images = np.expand_dims(images, axis = 3)
inner_region = img.get_center_region(inner_length, height, width)

model = mine.LogsiticRegression(images.shape[1:], model_settings)

val_start = int(images.shape[0] * float(1 / 7))
train_images = images[val_start:]
val_images = images[:val_start]

train_steps = int(np.ceil(train_images.shape[0] / batch_size))
val_steps = int(np.ceil(val_images.shape[0] / batch_size))

train_itr = mine.get_finite_dataset(train_images, inner_region, batch_size, loop = True)
val_itr = mine.cycle_generator(mine.get_finite_dataset, val_images, inner_region, batch_size)

print("Training model...")
model.train(train_itr, val_itr, train_steps, val_steps, max_epochs)

After the model has been trained, it can evaluate the MI of a dataset using two different equations. In _direct_ estimation, we compute the average log-ratio of $p(a, b)$ and $p(a)p(b)$ to get the MI. In _indirect_ estimation, we compute this same log-ratio but then subtract a quantity that should be zero if the MI estimate is exact, but will otherwise cancel some of the error in the direct estimate. Either approach may be preferred depending on the nature of the target dataset. The following method function of the `Model` class uses the trained model to compute both MI estimates:

In [ ]:
import numpy as np

def evaluate_MI(self, image_iterator, num_steps):
    cum_joint = 0
    cum_marginal = 0
    for (count, (image_batch, _)) in enumerate(image_iterator):
        [joint_outputs, marginal_outputs] = self.model.predict_on_batch(image_batch)
        cum_joint += np.mean(joint_outputs)
        cum_marginal += np.mean(np.exp(marginal_outputs))
        if count >= num_steps:
            break
    indirect_mi = cum_joint / num_steps - np.log(cum_marginal / num_steps)
    direct_mi = cum_joint / num_steps
    return (indirect_mi, direct_mi)

When the models are trained on center patches of different lengths, their estimates can be combined together to show how the MI scales with the size of the patch. The original paper did this by averaging over many trials at large sample sizes (offline, since it requires training hundreds of models); those pre-computed figures aren't included in this copy of the repo, so the cells below instead reproduce the equivalent curves live: first the exact analytic MI for all three field types (`get_area_law_prec`, `get_diffuse_volume_prec`, `get_sparse_volume_prec`) at both small and large correlation strengths, and then a live-trained neural estimate compared against the exact value for one of them, to check that the model estimates are able to closely match the exact MI.

### Reproducing the Gaussian scaling curves live

Unlike the empirical, neural-network-based MI estimates used elsewhere in this notebook, the MI of a Gaussian Markov random field has a closed form in terms of its covariance matrix (`image.get_analytic_MI`), so the exact curves can be computed instantly, with no training at all:

In [ ]:
from src import image as img

size = 27
lengths = list(range(1, size))
scaling_types = ["area", "diffuse", "sparse"]
strengths = ["small", "large"]

analytic_mi = {}
for scaling_type in scaling_types:
    for strength in strengths:
        (_, cov, _) = img.get_images(scaling_type, 1, strength = strength)
        image_length = int(cov.shape[0] ** 0.5)
        mi = img.get_analytic_MI(cov, [image_length, image_length], size + 1)
        analytic_mi[f"{scaling_type} ({strength})"] = mi[:-1]

img.plot_mi_scaling(analytic_mi, lengths = lengths, save_path = "gaussian_scaling_analytic.pdf")

### Validating the neural estimator against the exact value

Since we know the exact MI for these Gaussian fields, we can also use them to check that the neural-network estimator from earlier in this notebook is actually working: we sweep `mine.run_bipartition` across all partition lengths for one field (`diffuse`, `large` correlation — the same configuration used in the single-length training example above) using the same reduced settings as the real-dataset sweep, and plot its *direct* MI estimate against the exact analytic curve computed above.

In [ ]:
from src import mine, image as img

num_images = 10000
max_epochs = 30
eval_steps = 400

alg_settings = dict(
    image_type = "diffuse",
    num_images = num_images,
    strength = "large",
    algorithm = "logistic")
param_settings = dict(
    drop = 0,
    learn = 1e-4,
    layers = "[256, 256]",
    patience = 8,
    optm = "rms",
    val = 1 / 7,
    batch = 64,
    epoch = max_epochs)

neural_mi_values = []
for length in lengths:
    print(f"diffuse (large) - partition length: {length}")
    (indirect_mi, direct_mi) = mine.run_bipartition(length, alg_settings, param_settings, eval_steps = eval_steps)
    neural_mi_values.append(direct_mi)

img.plot_mi_scaling(
    {"diffuse (large) - analytic": analytic_mi["diffuse (large)"], "diffuse (large) - neural estimate": neural_mi_values},
    lengths = lengths,
    clip_negative = True,
    save_path = "gaussian_scaling_validation.pdf")

## MI scaling of real image datasets

Now that we have verified that the model is able to perform accurate MI estimation, we can carry out experiments on real image datasets. The real datasets available here are **MNIST** and **Fashion-MNIST** (simple $28 \times 28$ grayscale images), **CIFAR-10** (natural $32 \times 32$ colour images), and **LFW** (Labeled Faces in the Wild, a facial-recognition dataset). MNIST and Fashion-MNIST have relatively simple structure, while CIFAR-10 and LFW contain much richer natural-image and facial correlations.

Every dataset is conformed to the same $28 \times 28$ grayscale pipeline: CIFAR-10's colour images are converted to grayscale and centre-cropped from $32 \times 32$ to $28 \times 28$, and LFW's $125 \times 94$ images are resized to $28 \times 28$. The grayscale conversion uses the following weighted luminance coding:

In [ ]:
def convert_to_grayscale(images):
    (r, g, b) = (0.3, 0.59, 0.11)
    grayscale = r * images[..., 0] + g * images[..., 1] + b * images[..., 2]
    return grayscale

The neural network model introduced in the previous section can be trained on any of these real datasets in precisely the same manner as it was for the Gaussian Markov random fields. As an example, the following code block trains a model for MI estimation on MNIST:

In [ ]:
import numpy as np

from src import mine, image as img

num_images = 70000
inner_length = 10
batch_size = 64
max_epochs = 100
model_settings = dict(
    drop = 0, 
    learn = 1e-4, 
    layers = "[256, 256]", 
    patience = 20, 
    optm = "rms")

# Swap "mnist" for "fashion_mnist", "cifar10", or "lfw_faces" to use the
# other real datasets.
(images, _, _) = img.get_images("mnist", num_images)
(_, height, width) = images.shape
images = np.expand_dims(images, axis = 3)
inner_region = img.get_center_region(inner_length, height, width)

model = mine.LogsiticRegression(images.shape[1:], model_settings)

val_start = int(images.shape[0] * float(1 / 7))
train_images = images[val_start:]
val_images = images[:val_start]

train_steps = int(np.ceil(train_images.shape[0] / batch_size))
val_steps = int(np.ceil(val_images.shape[0] / batch_size))

train_itr = mine.get_finite_dataset(train_images, inner_region, batch_size, loop = True)
val_itr = mine.cycle_generator(mine.get_finite_dataset, val_images, inner_region, batch_size)

model.train(train_itr, val_itr, train_steps, val_steps, max_epochs)
model.evaluate_MI(val_itr, 5000)

As with the Gaussian distributions in the previous section, we can compute MI estimates for pixel patches of different sizes and plot how the MI changes. The original paper compared MNIST against the Tiny Images dataset this way; that pre-computed figure isn't included in this copy of the repo, and the Tiny Images dataset itself has been removed here in favor of the datasets listed above. In the original paper the difference in scaling behavior between the two datasets was stark, with the Tiny Images dataset showing a fairly linear pattern reminiscent of the nearest-neighbor Gaussian Markov random field, suggesting that local correlations are more dominant there than in MNIST. The same analysis is run live below across MNIST, Fashion-MNIST, CIFAR-10, and a facial dataset, to compare their correlation-scaling structure instead.

### Visualizing the scaling for a real dataset

To see the scaling pattern for real datasets ourselves, we repeat the MNIST training procedure above once per partition length **for every available real dataset** (MNIST, Fashion-MNIST, CIFAR-10, and LFW), using the `run_bipartition` helper from `src/mine.py` (the same function used by the `python -m src.mine` command-line script) to train a fresh model and evaluate its MI estimate at each length. The resulting curves are then plotted together with `image.plot_mi_scaling`, which reuses the exact figure style (font sizes, axis labels in nats, figure proportions) used to generate the reference figures shown above, so the output can be compared directly against them.

Two practical notes on the settings below, both taken from how this repo's own examples already describe running this cell:

- **Which MI estimate to plot.** `run_bipartition` returns both an *indirect* estimate (the Donsker-Varadhan-style bound used above) and a *direct* estimate (the plain average of the classifier's joint-sample logits). The indirect estimate involves a `log(mean(exp(...)))` term, which is much more sensitive to a handful of noisy/outlier batches and can dip below zero even though MI is analytically non-negative. The original paper's own real-dataset scaling figures (`plot_averages` in `src/image.py`) plot the *direct* estimate for exactly this reason, so we do the same here, and additionally clip any residual noise-driven negative values to zero when plotting (`clip_negative = True`).
- **Faster training.** Training 27 models per dataset (108 in total across 4 datasets) at the paper's full scale (70,000 images, up to 3,000 epochs — see `mine.ini`) would take a very long time. As this section's original note already advised, `num_images` and the epoch budget are reduced well below that here so the whole sweep finishes in a reasonable amount of time; increase them (up to the `mine.ini` values) for higher-fidelity curves closer to the paper's published figures, at the cost of much longer runtime.

In [ ]:
from src import mine, image as img

# Real datasets to compare. LFW (Labeled Faces in the Wild) is included as
# the facial dataset: it downloads automatically (via scikit-learn, no
# Kaggle account needed) and is large (13,000+ images across ~5,700 people).
image_types = ["mnist", "fashion_mnist", "cifar10", "lfw_faces"]

# Settings reduced from the paper's full scale (70,000 images, up to 3,000
# epochs) so the full sweep across partition lengths finishes quickly.
# Raise these (towards the mine.ini values) for higher-fidelity curves.
num_images = 10000
max_length = 28
max_epochs = 30
eval_steps = 400

param_settings = dict(
    drop = 0,
    learn = 1e-4,
    layers = "[256, 256]",
    patience = 8,
    optm = "rms",
    val = 1 / 7,
    batch = 64,
    epoch = max_epochs)

lengths = list(range(1, max_length))
direct_mi_by_dataset = {}
for image_type in image_types:
    try:
        img.get_images(image_type, 1)  # fail fast if this dataset isn't available
    except Exception as error:
        print(f"Skipping {image_type}: {error}\n")
        continue
    alg_settings = dict(
        image_type = image_type,
        num_images = num_images,
        strength = "small",
        algorithm = "logistic")
    direct_mi_values = []
    for length in lengths:
        print(f"{image_type} - partition length: {length}")
        (indirect_mi, direct_mi) = mine.run_bipartition(length, alg_settings, param_settings, eval_steps = eval_steps)
        direct_mi_values.append(direct_mi)
    direct_mi_by_dataset[image_type] = direct_mi_values

In [ ]:
img.plot_mi_scaling(direct_mi_by_dataset, lengths = lengths, clip_negative = True, save_path = "real_dataset_scaling.pdf")

LFW is now large enough (13,000+ images) that its MI should be a comparable magnitude to the other datasets rather than squashed flat like Olivetti Faces was; the cell below still plots it on its own axis to check.

In [ ]:
img.plot_mi_scaling(
    {"lfw_faces": direct_mi_by_dataset["lfw_faces"]},
    lengths = lengths,
    clip_negative = True,
    save_path = "lfw_faces_scaling.pdf")

### A real-data analog of the sparse/randomized boundary law

Very few real datasets actually exhibit the "sparse, randomized" correlation pattern tested synthetically above (`sparse`/`get_sparse_volume_prec`: nearest-neighbor correlations scattered to random positions) — real images almost always have their spatial correlations arranged locally and contiguously, which is exactly why MNIST/Fashion-MNIST/CIFAR-10/LFW all show roughly boundary-law-like scaling above rather than the sharper rise-then-crash shape of the synthetic `sparse` field. We can build a real-data analog directly from CIFAR-10 by permuting pixel *positions*, using the two new `image_type`s below:

- **`cifar10_shuffle_independent`**: a *different* random pixel permutation is applied to every image. This should destroy most of the *positional* correlation structure between the inner and outer patches, since a given pixel position no longer corresponds to a consistent original location from one image to the next.
- **`cifar10_shuffle_shared`**: the *same* random pixel permutation is applied to every image — directly analogous to how `get_sparse_volume_cov` permutes a nearest-neighbor precision matrix once and reuses that single permutation for every sample. Real pixel-to-pixel correlations are preserved (the same pair of original positions always maps to the same pair of scrambled positions across every image), just relocated to non-local, scattered positions instead of local ones.

One caveat worth noting before looking at the results: `cifar10_shuffle_independent` should *not* be expected to drive the MI all the way to zero. A pixel permutation preserves each image's own pixel *value* multiset (it only reorders positions), so this dataset can still leak a residual "do these two patches come from the same original image" signal through shared per-image statistics like overall brightness or contrast, even once all positional/spatial information has been scrambled away. `cifar10_shuffle_shared`, by contrast, should retain both that same residual signal *and* the real (now scattered) positional correlations, so its curve is expected to sit above `cifar10_shuffle_independent`'s.

In [ ]:
from src import mine, image as img

shuffle_types = ["cifar10_shuffle_independent", "cifar10_shuffle_shared"]
shuffle_mi_by_dataset = {}
for image_type in shuffle_types:
    alg_settings = dict(
        image_type = image_type,
        num_images = num_images,
        strength = "small",
        algorithm = "logistic")
    direct_mi_values = []
    for length in lengths:
        print(f"{image_type} - partition length: {length}")
        (indirect_mi, direct_mi) = mine.run_bipartition(length, alg_settings, param_settings, eval_steps = eval_steps)
        direct_mi_values.append(direct_mi)
    shuffle_mi_by_dataset[image_type] = direct_mi_values

img.plot_mi_scaling(
    {"cifar10 (unshuffled)": direct_mi_by_dataset["cifar10"], **shuffle_mi_by_dataset},
    lengths = lengths,
    clip_negative = True,
    save_path = "cifar10_shuffle_scaling.pdf")

Running this confirms both predictions: `cifar10_shuffle_independent` only rises to about 1.5-2 nats before flattening out (the residual per-image brightness/contrast signal, which saturates quickly since it doesn't need a large patch to estimate), while `cifar10_shuffle_shared` rises well *above* the unshuffled curve, peaking around 6.5-7 nats versus unshuffled CIFAR-10's ~3.7. That last part is worth explaining: scattering the same fixed permutation across every image spreads what were originally *local* (nearby-pixel) correlations out across the whole 28x28 grid. A contiguous inner/outer square cut through the *unshuffled* image only severs the relatively few correlated pairs that straddle its boundary, but the same cut through the *permuted* coordinate grid severs a much larger fraction of the (now scattered) correlated pairs — exactly the mechanism the synthetic `sparse` GMRF was built to demonstrate, now reproduced with real image statistics.